In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, log_loss, brier_score_loss,
    roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.optimize import minimize

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

# 1. Load Dataset
DATA_PATH = "data/cumulative.csv"
NASA_TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv"

if not os.path.exists(DATA_PATH):
    df = pd.read_csv(NASA_TAP_URL)
    df.to_csv(DATA_PATH, index=False)
else:
    df = pd.read_csv(DATA_PATH, comment="#")

df["target"] = (df["koi_disposition"] != "FALSE POSITIVE").astype(int)
y = df["target"].values
groups = df["kepid"].values

# 2. Target Leakage & Column Auditing
LEAKAGE_COLS = ["koi_disposition", "koi_pdisposition", "koi_score", "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]
ID_METADATA_COLS = ["rowid", "kepid", "kepoi_name", "kepler_name", "koi_tce_delivname", "koi_datalink_dvr", "koi_datalink_dvs", "koi_sparprov", "koi_vet_stat", "koi_vet_date", "koi_disp_prov", "koi_comment", "koi_fittype", "koi_limbdark_mod", "koi_parm_prov", "koi_trans_mod", "koi_quarters"]

drop_set = set(LEAKAGE_COLS + ID_METADATA_COLS + ["target"])
candidate_num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in drop_set and df[c].isna().mean() < 0.50]

X_clean = df[candidate_num_cols].copy()
expected_depth_ppm = ((X_clean["koi_prad"] / (109.076 * X_clean["koi_srad"])) ** 2) * 1e6
X_clean["eng_depth_radius_consistency"] = X_clean["koi_depth"] / (expected_depth_ppm + 1.0)
X_clean["eng_duty_cycle"] = (X_clean["koi_duration"] / 24.0) / (X_clean["koi_period"] + 1e-6)

SKEW_COLS = ["koi_period", "koi_depth", "koi_prad", "koi_insol", "koi_model_snr", "koi_duration"]
for col in SKEW_COLS:
    if col in X_clean.columns:
        X_clean[f"log_{col}"] = np.log1p(np.clip(X_clean[col], a_min=0, a_max=None))

# 3. Host-Star Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
dev_idx, test_idx = next(gss.split(X_clean, y, groups=groups))
X_dev, y_dev, groups_dev = X_clean.iloc[dev_idx].reset_index(drop=True), y[dev_idx], groups[dev_idx]
X_test, y_test, groups_test = X_clean.iloc[test_idx].reset_index(drop=True), y[test_idx], groups[test_idx]

# 4. Calibration Helper
def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    binids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for i in range(n_bins):
        mask = binids == i
        if np.any(mask):
            ece += np.abs(np.mean(y_true[mask]) - np.mean(y_prob[mask])) * (np.sum(mask) / len(y_true))
    return ece

def fit_temperature_scaling(logits, y_true):
    def bce_loss(T):
        t = T[0]
        p = 1.0 / (1.0 + np.exp(-np.clip(logits / t, -50, 50)))
        p = np.clip(p, 1e-7, 1.0 - 1e-7)
        return -np.mean(y_true * np.log(p) + (1.0 - y_true) * np.log(1.0 - p))
    res = minimize(bce_loss, x0=[1.0], bounds=[(0.05, 10.0)], method="L-BFGS-B")
    return res.x[0]

# 5. ResNet-MLP Architecture
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.25):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(x + self.fc(x))

class ExoplanetResNetMLP(nn.Module):
    def __init__(self, in_features, hidden_dim=128, dropout=0.25):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU())
        self.block1 = ResidualBlock(hidden_dim, dropout)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        return self.head(self.block1(self.stem(x))).squeeze(-1)

def train_resnet_fold(X_tr, y_tr, X_va, y_va, epochs=50, batch_size=128, lr=1e-3):
    prep = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("scaler", StandardScaler())])
    X_tr_s, X_va_s = prep.fit_transform(X_tr), prep.transform(X_va)
    model = ExoplanetResNetMLP(X_tr_s.shape[1], hidden_dim=128, dropout=0.25)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32)), batch_size=batch_size, shuffle=True)
    
    best_va_loss, best_va_logits = float("inf"), None
    for epoch in range(epochs):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            va_logits = model(torch.tensor(X_va_s, dtype=torch.float32))
            va_loss = criterion(va_logits, torch.tensor(y_va, dtype=torch.float32)).item()
        if va_loss < best_va_loss:
            best_va_loss = va_loss
            best_va_logits = va_logits.numpy()
    return best_va_logits

# 6. Cross-Validation
gkf = GroupKFold(n_splits=5)
classical_models = {
    "Logistic Regression (L2, C=0.1)": Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("scaler", StandardScaler()), ("clf", LogisticRegression(C=0.1, max_iter=1000, random_state=SEED))]),
    "Random Forest (500 trees, depth=16)": Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("clf", RandomForestClassifier(n_estimators=500, max_depth=16, max_features="sqrt", n_jobs=-1, random_state=SEED))]),
    "XGBoost (Tuned Classical Baseline)": XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, reg_lambda=5.0, gamma=0.5, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", n_jobs=-1, random_state=SEED)
}

oof_probs = {name: np.zeros(len(X_dev)) for name in classical_models}
oof_probs["ResNet-MLP (Temp. Scaled)"] = np.zeros(len(X_dev))

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_dev, y_dev, groups=groups_dev)):
    X_tr, y_tr = X_dev.iloc[tr_idx], y_dev[tr_idx]
    X_va, y_va = X_dev.iloc[va_idx], y_dev[va_idx]
    for name, model in classical_models.items():
        model.fit(X_tr, y_tr)
        oof_probs[name][va_idx] = model.predict_proba(X_va)[:, 1]
    
    va_logits = train_resnet_fold(X_tr, y_tr, X_va, y_va)
    half = len(va_idx) // 2
    T_opt = fit_temperature_scaling(va_logits[:half], y_va[:half])
    cal_p = 1.0 / (1.0 + np.exp(-np.clip(va_logits / T_opt, -50, 50)))
    oof_probs["ResNet-MLP (Temp. Scaled)"][va_idx] = cal_p

# Print Metrics Summary
rows = []
for name, probs in oof_probs.items():
    preds = (probs >= 0.5).astype(int)
    rows.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_dev, probs),
        "PR-AUC": average_precision_score(y_dev, probs),
        "F1": f1_score(y_dev, preds),
        "ECE": compute_ece(y_dev, probs)
    })
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Save Figures
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for name in oof_probs.keys():
    p = oof_probs[name]
    fpr, tpr, _ = roc_curve(y_dev, p)
    prec, rec, _ = precision_recall_curve(y_dev, p)
    axes[0].plot(fpr, tpr, lw=2, label=f"{name.split(' (')[0]} (AUC={roc_auc_score(y_dev, p):.3f})")
    axes[1].plot(rec, prec, lw=2, label=f"{name.split(' (')[0]} (PR={average_precision_score(y_dev, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_title("ROC Curves")
axes[1].set_title("Precision-Recall Curves")
plt.tight_layout()
plt.savefig("figures/fig1_roc_pr_curves.png", dpi=300)

plt.figure(figsize=(5.5, 4.4))
plt.plot([0, 1], [0, 1], "k--", label="Perfect")
for name in oof_probs.keys():
    pt, pp = calibration_curve(y_dev, oof_probs[name], n_bins=10)
    plt.plot(pp, pt, marker="o", lw=2, label=f"{name} (ECE={compute_ece(y_dev, oof_probs[name]):.3f})")
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Reliability Diagram")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("figures/fig2_diagnostics_calibration.png", dpi=300)
print("\nGenerated and saved figures successfully!")

: 